# Notebook 2 - Speech Recognition and NLP

Roadmap role: Week 2. Literature-driven rule: build labeled data, keep ASR and intent extraction separate, train or fine-tune intent extraction only with documented splits.

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
COMMANDS = ROOT / 'datasets' / 'commands'
AUDIO = ROOT / 'datasets' / 'sample_audio'
print('commands_dir', COMMANDS.exists())
print('audio_dir', AUDIO.exists())
print('Training uses labeled text commands. ASR is evaluated separately when WAV files exist.')


## Train the initial intent baseline

This uses a synthetic labeled dataset and a lightweight supervised baseline. The result is not a real-user benchmark. Run both `v0` and `v1` so the weak baseline and improved feature configuration remain comparable.

In [ ]:
!python scripts/train_intent_model.py \
  --dataset datasets/commands/intent_labeled_synthetic.jsonl \
  --model-output outputs/model_artifacts/intent_nb_v0.json \
  --metrics-output outputs/evaluations/intent_nb_v0_metrics.json \
  --comparison-output outputs/evaluations/intent_nb_v0_vs_deterministic.json \
  --validation-output outputs/evaluations/intent_nb_v0_validation_metrics.json \
  --seed 17


In [ ]:
!python scripts/train_intent_model.py \
  --dataset datasets/commands/intent_labeled_synthetic.jsonl \
  --model-output outputs/model_artifacts/intent_nb_v1.json \
  --metrics-output outputs/evaluations/intent_nb_v1_metrics.json \
  --comparison-output outputs/evaluations/intent_nb_v1_vs_deterministic.json \
  --validation-output outputs/evaluations/intent_nb_v1_validation_metrics.json \
  --seed 17 \
  --model-name trained_nb_v1 \
  --model-version 0.2 \
  --include-bigrams \
  --include-alias-features \
  --alias-feature-weight 3


In [ ]:
import json
for name in [
    'intent_nb_v0_validation_metrics.json',
    'intent_nb_v0_metrics.json',
    'intent_nb_v1_validation_metrics.json',
    'intent_nb_v1_metrics.json',
]:
    metrics_path = ROOT / 'outputs' / 'evaluations' / name
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text())
        print(name, metrics['summary'])
    else:
        print(name, 'missing; run the training cells first.')


## Evaluate a frozen model

Use this for validation, test, human-written, or gold-transcript datasets after a model artifact has already been trained.

In [ ]:
!python scripts/evaluate_intent_model.py \
  --model outputs/model_artifacts/intent_nb_v1.json \
  --dataset datasets/commands/intent_labeled_synthetic.jsonl \
  --split validation \
  --output outputs/evaluations/intent_nb_v1_saved_model_validation.json


## Train on curated manual Week 2 commands

Run this only after `datasets/commands/human_written_commands_curated_v1.jsonl` exists. This dataset is assistant-curated from manual commands, not an ASR benchmark.

In [ ]:
curated = COMMANDS / 'human_written_commands_curated_v1.jsonl'
if curated.exists():
    !python scripts/train_intent_model.py \
      --dataset datasets/commands/human_written_commands_curated_v1.jsonl \
      --model-output outputs/model_artifacts/intent_nb_human_curated_v1.json \
      --metrics-output outputs/evaluations/intent_nb_human_curated_v1_metrics.json \
      --comparison-output outputs/evaluations/intent_nb_human_curated_v1_vs_deterministic.json \
      --validation-output outputs/evaluations/intent_nb_human_curated_v1_validation_metrics.json \
      --seed 17 \
      --model-name trained_nb_human_curated_v1 \
      --model-version 0.1 \
      --include-bigrams \
      --include-alias-features \
      --alias-feature-weight 3
    !python scripts/train_intent_model.py \
      --dataset datasets/commands/human_written_commands_curated_v1.jsonl \
      --model-output outputs/model_artifacts/intent_nb_human_curated_v2.json \
      --metrics-output outputs/evaluations/intent_nb_human_curated_v2_metrics.json \
      --comparison-output outputs/evaluations/intent_nb_human_curated_v2_vs_deterministic.json \
      --validation-output outputs/evaluations/intent_nb_human_curated_v2_validation_metrics.json \
      --seed 17 \
      --model-name trained_nb_human_curated_v2 \
      --model-version 0.2 \
      --include-bigrams \
      --include-alias-features \
      --alias-feature-weight 3 \
      --use-rule-overrides
else:
    print('No curated manual command dataset yet.')


## Validate span labels for slot extraction

Run this after a human-verified span dataset exists. This produces BIO token labels for future spaCy or Hugging Face token-classification experiments.

In [ ]:
span_dataset = COMMANDS / 'human_verified_span_commands.jsonl'
if span_dataset.exists():
    !python scripts/validate_span_dataset.py \
      --dataset datasets/commands/human_verified_span_commands.jsonl \
      --summary-output outputs/evaluations/human_verified_span_commands_summary.json \
      --bio-output outputs/evaluations/human_verified_span_commands_bio.jsonl
else:
    print('No human-verified span dataset yet. See docs/week2_span_annotation.md before creating one.')


## Train the initial BIO span tagger

This is a dependency-free supervised baseline over human-verified span labels. It is not a final spaCy or Hugging Face model.

In [ ]:
if span_dataset.exists():
    !python scripts/train_span_tagger.py \
      --dataset datasets/commands/human_verified_span_commands.jsonl \
      --model-output outputs/model_artifacts/span_nb_v0.json \
      --metrics-output outputs/evaluations/span_nb_v0_metrics.json \
      --validation-output outputs/evaluations/span_nb_v0_validation_metrics.json \
      --seed 17 \
      --model-name span_nb_v0 \
      --model-version 0.1
    !python scripts/train_span_tagger.py \
      --dataset datasets/commands/human_verified_span_commands.jsonl \
      --model-output outputs/model_artifacts/span_nb_v1.json \
      --metrics-output outputs/evaluations/span_nb_v1_metrics.json \
      --validation-output outputs/evaluations/span_nb_v1_validation_metrics.json \
      --seed 17 \
      --model-name span_nb_v1 \
      --model-version 0.2 \
      --use-transitions
else:
    print('No human-verified span dataset yet.')


## Export Hugging Face token-classification data

This creates split JSONL files and a stable label map for transformer fine-tuning.

In [ ]:
if span_dataset.exists():
    !python scripts/export_hf_token_dataset.py \
      --dataset datasets/commands/human_verified_span_commands.jsonl \
      --output-dir outputs/hf_token_dataset
else:
    print('No human-verified span dataset yet.')


## Fine-tune a Hugging Face token classifier

Run this in Colab with a T4 GPU runtime when you are ready to train a transformer baseline. Record the model name, seed, epochs, learning rate, runtime, package versions, and outputs. Do not report results unless the raw metrics files are saved.

In [ ]:
# Colab dependency setup:
# %pip install -q "transformers[torch]" accelerate datasets seqeval

# Verify Colab is using a GPU runtime before training:
# import torch
# assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU"
# print(torch.cuda.get_device_name(0))

# After installing dependencies, run:
# !python scripts/train_hf_token_classifier.py \
#   --dataset-dir outputs/hf_token_dataset \
#   --pretrained-model distilbert-base-uncased \
#   --output-dir outputs/model_artifacts/hf_token_classifier_distilbert \
#   --metrics-output outputs/evaluations/hf_token_classifier_distilbert_metrics.json \
#   --validation-output outputs/evaluations/hf_token_classifier_distilbert_validation_metrics.json \
#   --epochs 5 \
#   --learning-rate 0.00002 \
#   --batch-size 8 \
#   --seed 17 \
#   --required-device-substring T4


## Validate real Week 2 collection files

Run these after real human-written commands or real WAV manifests exist. Do not treat the synthetic dataset as human or speech data.

In [ ]:
# Fast path for a real sample after you upload a WAV to Colab:
# !python scripts/collect_week2_sample.py \
#   --text "Send two drones north and inspect the crops." \
#   --wav "/content/your_recording.wav" \
#   --split train

!python scripts/validate_command_dataset.py \
  --dataset datasets/commands/intent_labeled_synthetic.jsonl \
  --summary-output outputs/evaluations/intent_labeled_synthetic_summary.json \
  --require-splits train,validation,test


In [ ]:
audio_manifest = AUDIO / 'manifest.jsonl'
if audio_manifest.exists():
    !python scripts/validate_audio_manifest.py \
      --manifest datasets/sample_audio/manifest.jsonl \
      --dataset-root . \
      --summary-output outputs/evaluations/audio_manifest_summary.json
else:
    print('No audio manifest yet. Add real WAV files and datasets/sample_audio/manifest.jsonl before ASR evaluation.')
